# 04 — Baseline Models

**Goal:** compare simple, reproducible baselines using chronological validation and metrics suited to rare maintenance warnings.

Models: most-frequent dummy, class-weighted logistic regression, and class-weighted random forest. Choose the decision threshold on validation data only; report the test result once.

In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
FEATURE_PATH = PROJECT_ROOT / 'data' / 'processed' / 'tina_features.parquet'
TARGET = 'maintenance_within_horizon'
TIME_COL = 'unix'  # verified TiNA source time field
RANDOM_SEED = 613

## Load the leakage-audited feature table

In [2]:
if not FEATURE_PATH.exists():
    raise FileNotFoundError('Run notebook 03 and save the audited local feature table first.')
table = pd.read_parquet(FEATURE_PATH).sort_values(TIME_COL).reset_index(drop=True)
feature_columns = [column for column in table.columns if column not in {TARGET, TIME_COL}]
X = table[feature_columns]
y = table[TARGET].astype('int8')
print(f'Rows: {len(table):,}; features: {len(feature_columns)}; positive rate: {y.mean():.4%}')

Rows: 257,400; features: 66; positive rate: 6.1336%


## Chronological train/validation/test split

In [3]:
train_end = int(len(table) * 0.70)
validation_end = int(len(table) * 0.85)
X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_validation, y_validation = X.iloc[train_end:validation_end], y.iloc[train_end:validation_end]
X_test, y_test = X.iloc[validation_end:], y.iloc[validation_end:]

pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'rows': [len(y_train), len(y_validation), len(y_test)],
    'positive_rate': [y_train.mean(), y_validation.mean(), y_test.mean()],
    'start': [table[TIME_COL].iloc[0], table[TIME_COL].iloc[train_end], table[TIME_COL].iloc[validation_end]],
    'end': [table[TIME_COL].iloc[train_end - 1], table[TIME_COL].iloc[validation_end - 1], table[TIME_COL].iloc[-1]],
})

,split,rows,positive_rate,start,end
0,train,180180,0.053152,1504828800,1505008979
1,validation,38610,0.160865,1505008980,1505047589
2,test,38610,0.000000,1505047590,1505086199


## Fit baselines

In [4]:
linear_preprocessing = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
tree_preprocessing = SimpleImputer(strategy='median')

models = {
    'dummy_most_frequent': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', DummyClassifier(strategy='most_frequent')),
    ]),
    'logistic_balanced': Pipeline([
        ('preprocess', linear_preprocessing),
        ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_SEED)),
    ]),
    'random_forest_balanced': Pipeline([
        ('imputer', tree_preprocessing),
        ('model', RandomForestClassifier(
            n_estimators=200, max_depth=12, min_samples_leaf=20,
            class_weight='balanced_subsample', n_jobs=-1, random_state=RANDOM_SEED,
        )),
    ]),
}

for name, model in models.items():
    print(f'Fitting {name} ...')
    model.fit(X_train, y_train)

Fitting dummy_most_frequent ...


Fitting logistic_balanced ...


/Users/ganeshmishra/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/ganeshmishra/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/ganeshmishra/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/ganeshmishra/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/ganeshmishra/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features

Fitting random_forest_balanced ...


## Select thresholds on validation data

In [5]:
def positive_scores(model, features):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(features)[:, 1]
    return model.predict(features).astype(float)

def choose_f1_threshold(labels, scores):
    thresholds = np.linspace(0.01, 0.99, 99)
    values = [f1_score(labels, scores >= threshold, zero_division=0) for threshold in thresholds]
    return float(thresholds[int(np.argmax(values))])

thresholds = {}
for name, model in models.items():
    validation_scores = positive_scores(model, X_validation)
    thresholds[name] = 0.5 if name.startswith('dummy') else choose_f1_threshold(y_validation, validation_scores)
thresholds

/Users/ganeshmishra/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/ganeshmishra/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/ganeshmishra/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


{'dummy_most_frequent': 0.5,
 'logistic_balanced': 0.01,
 'random_forest_balanced': 0.01}

## Final test metrics

In [6]:
def evaluate(name, model, features, labels, threshold):
    scores = positive_scores(model, features)
    predictions = (scores >= threshold).astype('int8')
    unique_scores = np.unique(scores)
    return {
        'model': name,
        'threshold': threshold,
        'precision': precision_score(labels, predictions, zero_division=0),
        'recall': recall_score(labels, predictions, zero_division=0),
        'f1': f1_score(labels, predictions, zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(labels, predictions),
        'pr_auc': average_precision_score(labels, scores) if len(unique_scores) > 1 else np.nan,
        'roc_auc': roc_auc_score(labels, scores) if len(unique_scores) > 1 else np.nan,
    }

test_results = pd.DataFrame([
    evaluate(name, model, X_test, y_test, thresholds[name])
    for name, model in models.items()
]).sort_values('pr_auc', ascending=False)
test_results

/Users/ganeshmishra/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Users/ganeshmishra/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/ganeshmishra/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/ganeshmishra/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/ganeshmishra/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/ganeshmishra/L

,model,threshold,precision,recall,f1,balanced_accuracy,pr_auc,roc_auc
1,logistic_balanced,0.01,0.0,0.0,0.0,0.287102,0.0,NaN
2,random_forest_balanced,0.01,0.0,0.0,0.0,0.501243,0.0,NaN
0,dummy_most_frequent,0.50,0.0,0.0,0.0,1.000000,NaN,NaN


## Operational evaluation still required

Row-level metrics can exaggerate performance when many adjacent rows describe one event. Add event grouping and report: percentage of maintenance events warned in advance, median warning lead time, false warning episodes per operating day, warning duration, and performance by maintenance type. Compare thresholds against the operational cost of missed events and unnecessary inspections.

## Optional small results export

In [7]:
RESULT_PATH = PROJECT_ROOT / 'results' / 'baseline_metrics.csv'
# test_results.to_csv(RESULT_PATH, index=False)
print(f'After final review, uncomment to save: {RESULT_PATH}')
# Keep fitted models local; model formats are ignored by Git.

After final review, uncomment to save: /Users/ganeshmishra/Documents/Codex/2026-09-19/referenced-chatgpt-conversation-this-is-an/outputs/ai-based-predictive-maintenance-mining-equipment/results/baseline_metrics.csv
